# 02 · Validate the Fallen-Angel Factor

Phase 3 verification: does buying IG→HY downgrades and hedging with HYG earn excess return over 12 months?

1. Load the curated 2015–2024 downgrade events (`config/fallen_angel_events.yaml` — **dates flagged TODO, confirm via RATC<GO>**).
2. Run `run_backtest`: per event, equal-weight bond TR proxy vs HYG total return.
3. Summary stats overall and stratified by **sector** and **catalyst**.
4. Plotly: excess distribution, per-catalyst boxes, and event excess paths.

Run from `fallen_angels/` with a logged-in Terminal. No notebook magics are used.

In [ ]:
import logging

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from fallen_angels.config import load_config
from fallen_angels import backtest as bt

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

ISSUER = "CNC"  # only supplies Bloomberg field names + the hedge instrument
cfg = load_config(ISSUER)
events = bt.load_events()
pd.DataFrame([e.model_dump() for e in events])[[
    "issuer", "ticker", "sector", "catalyst", "downgrade_date", "from_rating", "to_rating"
]]

## 1. Run the study

`hedge_ratio=1.0` for the beta-neutral factor read; rerun with `0.6` to mirror the live trade's hedge band. Failed events (chains that don't resolve, etc.) get an `error` string instead of aborting.

In [ ]:
results, paths = bt.run_backtest(
    events, cfg, horizon_days=365, entry_lag_days=0, hedge_ratio=1.0, return_paths=True
)
failed = results[results["error"].notna()]
if not failed.empty:
    print("Failed events — fix via explicit `bonds:` in the events YAML:")
    print(failed[["ticker", "event_date", "error"]].to_string(index=False))
results.round(4)

## 2. Summary: overall, by sector, by catalyst

`sharpe` is cross-sectional (mean/std of per-event 12M excess) — already annual, no further scaling.

In [ ]:
display_cols = lambda df: df.round(3)
print("=== Overall ===")
print(display_cols(bt.summarize(results)).to_string())
print("\n=== By sector ===")
print(display_cols(bt.summarize(results, by="sector")).to_string())
print("\n=== By catalyst ===")
print(display_cols(bt.summarize(results, by="catalyst")).to_string())

## 3. Distribution of 12M excess returns

In [ ]:
valid = results.dropna(subset=["excess"])
fig = px.histogram(
    valid, x="excess", color="catalyst", nbins=20, marginal="rug",
    hover_data=["ticker", "event_date"],
    title="12M excess return vs HYG — fallen-angel events 2015–2024",
)
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(height=420, xaxis_tickformat=".0%")
fig.show()

fig2 = px.box(
    valid, x="catalyst", y="excess", points="all", hover_data=["ticker"],
    title="Excess return by downgrade catalyst",
)
fig2.add_hline(y=0, line_dash="dash", line_color="gray")
fig2.update_layout(height=420, yaxis_tickformat=".0%")
fig2.show()

## 4. Event excess paths (hedged cumulative return from the downgrade)

The shape matters for the live trade: how long does convergence take, and how deep is the interim drawdown the position must survive?

In [ ]:
fig3 = go.Figure()
for ticker, path in paths.items():
    days = (path.index - path.index[0]).days
    fig3.add_trace(go.Scatter(x=days, y=path.values, name=ticker, mode="lines", opacity=0.7))
fig3.add_hline(y=0, line_dash="dash", line_color="gray")
fig3.update_layout(
    title="Hedged excess-return paths, days since downgrade",
    xaxis_title="Days since downgrade", yaxis_title="Cumulative excess return",
    yaxis_tickformat=".0%", height=520,
)
fig3.show()

## 5. Sensitivity: live-trade hedge ratio

Re-run at the live band midpoint (0.6) — the factor should still clear zero if the thesis isn't just long HY beta.

In [ ]:
results_060 = bt.run_backtest(events, cfg, horizon_days=365, hedge_ratio=0.6)
print(bt.summarize(results_060).round(3).to_string())